In [371]:
import pandas as pd
import numpy as np
import statistics
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize
from sklearn.linear_model import HuberRegressor

In [372]:
F1 = open('ROVER1.GK', 'r')

t = []
x = []
y = []
z = []

def unp(F):
    t.clear()
    x.clear()
    y.clear()
    z.clear()
    for line in F: 
        data = line.split()
        t.append(float(data[0]))
        x.append(float(data[1]))
        y.append(float(data[2]))
        z.append(float(data[3]))
    a = pd.DataFrame({"t":t,"x":x,"y":y,"z":z})
    return a

l = 456  # 456 647 786 974
r = 646  # 646 785 973 1094
r0 = unp(F1)
r1 = r0[l:r+1]
t = r1["t"]-t[l]
x = r1["x"]-x[l]
y = r1["y"]-y[l]
z = r1["z"]-z[l]

In [373]:
# Этот блок нужен для того, чтобы МНМ работал... (меняем формат данных)

t1 = []
vx = []
vy = []
vz = []

for i in range(l+1,r+1):    # создаю эти массивы, чтобы вытащить зависимость скоростей на участках от времени
    t1.append(int(t[i]))
    vx.append(x[i]/t[i])
    vy.append(y[i]/t[i])
    vz.append(z[i]/t[i])
    #print(t1[i-l-1], vx[i-l-1], vy[i-l-1], vz[i-l-1])

b = pd.DataFrame({"t":t1,"x":vx,"y":vy,"z":vz})
t1.clear()
vx.clear()
vy.clear()
vz.clear()
t1 = b["t"]
vx = b["x"]
vy = b["y"]
vz = b["z"]

In [374]:
# 1. Метод наименьших квадратов (МНК)

def mnk(x, y):
    # Добавление столбца единиц для свободного члена
    X = np.vstack([x, np.ones(len(x))]).T
    # Рассчитываем коэффициенты с помощью псевдообратной матрицы
    beta = np.linalg.inv(X.T @ X) @ X.T @ y
    return beta

def f():

    a0, a1 = mnk(t1,vx)     # применяем МНК, чтобы найти те самые зависимости проекций скоростей от времени
    b0, b1 = mnk(t1,vy)
    c0, c1 = mnk(t1,vz)

    v_x = a0*t1[r-l-1]        # проекции средней скорости
    v_y = b0*t1[r-l-1]
    v_z = c0*t1[r-l-1]

    print(f"Средняя скорость вдоль стороны прямоугольника: {np.sqrt(v_x**2 + v_y**2 + v_z**2):.4f} м/с = {np.sqrt(v_x**2 + v_y**2 + v_z**2)*36:.4f} км/ч")

    #x1 = a0*t[r] + a1
    #y1 = b0*t[r] + b1
    #z1 = c0*t[r] + c1

f()

Средняя скорость вдоль стороны прямоугольника: 0.3693 м/с = 13.2933 км/ч


In [375]:
# 2. Метод наименьших модулей (МНМ)

def mnmx(params):
    i, j = params
    residuals = vx - (i + j * t1)
    return np.sum(np.abs(residuals))

def mnmy(params):
    i, j = params
    residuals = vy - (i + j * t1)
    return np.sum(np.abs(residuals))

def mnmz(params):
    i, j = params
    residuals = vz - (i + j * t1)
    return np.sum(np.abs(residuals))

def g():

    result = minimize(mnmx, [-1000, -1000])
    a1, a0 = result.x
    result = minimize(mnmy, [-1000, -1000])
    b1, b0 = result.x
    result = minimize(mnmz, [-1000, -1000])
    c1, c0 = result.x

    v_x = a0*t1[r-l-1]        # проекции средней скорости
    v_y = b0*t1[r-l-1]
    v_z = c0*t1[r-l-1]

    print(f"Средняя скорость вдоль стороны прямоугольника: {np.sqrt(v_x**2 + v_y**2 + v_z**2):.4f} м/с = {np.sqrt(v_x**2 + v_y**2 + v_z**2)*36:.4f} км/ч")

    #x_pred = a0*t + a1
    #y_pred = b0*t + b1
    #z_pred = c0*t + c1

g()

Средняя скорость вдоль стороны прямоугольника: 0.2352 м/с = 8.4663 км/ч
